# 09 — V2 feature stability & frozen comparison

Two questions, one notebook:

1. **Are the V2 LLM count features stable?** Re-run the *identical* V2 prompt with a second,
   independent model (`qwen3.6-35b` vs the original `qwen3.5:122b`) and measure per-feature
   agreement (Spearman ρ, ICC(2,1), exact-match).
2. **Do V2 features add information beyond simple surface features?** Compare V2 against
   transcript length, surface statistics, V1 categorical features and a TF-IDF ceiling under
   the same repeated-CV protocol as notebooks 04–08, now with **paired** bootstrap tests.

Automatic gates (no eyeballing — this is for the methods section):

* **Degeneracy gate** — a count feature is dropped if `zero_share > 0.70` on the labelled set
  in *either* run.
* **Stability flag** — a feature is "stable" if Spearman ρ(run A, run B) ≥ 0.60. Reported for
  every feature; a stable-only sensitivity arm is included.

The pipeline (features → preprocessing → classifier → params) is **frozen** to disk at the end.
The 61-case test set is **not** touched here.

## Block 0 — configuration

In [1]:
import json, time, hashlib, datetime, warnings
from pathlib import Path

import numpy as np
import pandas as pd

from helpers import (
    MODEL_A, MODEL_B, BASE_URL, TEMPERATURE, MAX_TOKENS, SEED, N_CALIB,
    ZERO_SHARE_MAX, STABILITY_MIN_RHO, EXPECTED_PROMPT_SHA,
)

warnings.filterwarnings("ignore")

# helpers.py (one flat module next to the notebooks) holds the endpoint/key,
# model names, gate thresholds, the frozen V2 prompt, StreamingLocalProvider and
# the CV/bootstrap core. Everything else is inlined in the cell that uses it.

RUN_LLM = False              # both v2_raw.jsonl + v2b_raw.jsonl are cached -> analysis-only re-run
                             # (set True to re-extract with MODEL_B from scratch)

DATA       = Path("fileDataset")
OUT        = DATA / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
FROZEN_DIR = OUT / "frozen_pipeline_v2"
FROZEN_DIR.mkdir(parents=True, exist_ok=True)
RAW_A      = OUT / "v2_raw.jsonl"           # produced by notebook 08 (MODEL_A)
RAW_B      = OUT / "v2b_raw.jsonl"          # produced here (MODEL_B)
META_B     = OUT / "v2b_run_metadata.json"

assert RAW_A.exists(), f"{RAW_A} not found -- run notebook 08 first"
print("MODEL_A :", MODEL_A, "| cached :", RAW_A, "|", RAW_A.exists())
print("MODEL_B :", MODEL_B, "| target :", RAW_B, "|", RAW_B.exists())
print("gates   : zero_share_max", ZERO_SHARE_MAX, "| stability_min_rho", STABILITY_MIN_RHO)
print("RUN_LLM :", RUN_LLM)


MODEL_A : qwen3.5:122b | cached : fileDataset/outputs/v2_raw.jsonl | True
MODEL_B : qwen3.6-35b | target : fileDataset/outputs/v2b_raw.jsonl | True
gates   : zero_share_max 0.7 | stability_min_rho 0.6
RUN_LLM : False


## Block 1 — the V2 prompt (byte-identical to notebook 08)

The stability comparison is only meaningful if the prompt is exactly the same. The SHA is
asserted against the value recorded in `run_metadata.json`.

In [2]:
from helpers import V2_PROMPT, PROMPT_SHA, assert_prompt_hash, assert_domain_blind

print(f"prompt sha256[:16] = {PROMPT_SHA}  ({len(V2_PROMPT)} chars)")
# two extraction runs are only comparable under a byte-identical prompt; a frozen
# pipeline is only valid under the prompt it was built with. Re-assert here even
# though helpers.py single-sources the text -- this also catches a stale import.
assert_prompt_hash(EXPECTED_PROMPT_SHA)
assert_domain_blind()
print("prompt matches run A, domain-blind check: OK")


prompt sha256[:16] = f52aae5d939d620e  (2428 chars)
prompt matches run A, domain-blind check: OK


## Block 2 — load transcripts + surface statistics

`train/negative` (0), `train/positive` (1), `overview/` (unlabelled). `test/` is never read.

In [3]:
import re

if not DATA.exists():
    raise FileNotFoundError(f"dataset not found at {DATA}")

# train/negative -> 0, train/positive -> 1, overview/ -> unlabelled. test/ is never read.
def read_split(folder, label):
    return [{"file": f.name, "label": label,
             "text": f.read_text(encoding="utf-8", errors="replace").strip()}
            for f in sorted(folder.glob("*.txt"))]

SPLITS = [("train/negative", 0), ("train/positive", 1), ("overview", np.nan)]
docs = pd.DataFrame([r for sub, lab in SPLITS for r in read_split(DATA / sub, lab)])
assert docs.file.is_unique and not docs.empty

# surface statistics -- the length confound every LLM feature is measured against.
# n_word: whitespace tokens floored at 1 (NOT \w+); n_sent: [.!?]+ runs floored at 1.
_WORD = re.compile(r"\w+", re.UNICODE)
def surface(t):
    nw = max(1, len(t.split()))
    nt = len(set(_WORD.findall(t.lower())))
    ns = max(1, len(re.findall(r"[.!?]+", t)))
    return {"n_char": len(t), "n_word": nw, "n_type": nt, "ttr": nt / nw,
            "n_sent": ns, "mlu": nw / ns, "n_comma": t.count(",")}

docs = pd.concat([docs, docs.text.apply(lambda t: pd.Series(surface(t)))], axis=1)
SURFACE = ["n_word", "n_type", "ttr", "n_sent", "mlu", "n_comma", "n_char"]

n_lab, n_unlab = int(docs.label.notna().sum()), int(docs.label.isna().sum())
assert (n_lab, n_unlab) == (241, 86), (n_lab, n_unlab)
print(f"{len(docs)} docs -> {n_lab} labelled + {n_unlab} unlabelled")
print(docs.loc[docs.label.notna(), "label"].astype(int)
          .map({0: "negative", 1: "positive"}).value_counts().to_string())


327 docs -> 241 labelled + 86 unlabelled
label
negative    171
positive     70


## Block 3 — provider (identical `StreamingLocalProvider` as notebook 08)

Streaming + `think:false` + retries on transient failures. Only the target model changes.

In [4]:
from helpers import make_provider

# StreamingLocalProvider (streaming + think-off + transient-failure retries),
# defined once in helpers.py and shared with notebook 08.
provider = make_provider(MODEL_B)
print("provider ready:", type(provider).__name__, "| model:", provider.text_model)


provider ready: StreamingLocalProvider | model: qwen3.6-35b


## Block 4 — calibration on MODEL_B (timing + groundedness gate)

In [5]:
# groundedness / missing-fields QC -- inlined (short, notebook-specific)
EVIDENCE_FIELDS = ["named_entities", "specific_action_verbs", "generic_verbs",
                   "locative_expressions", "hedge_spans", "deictic_spans",
                   "metacomment_spans", "diminutive_or_affective_forms", "quantity_expressions"]
REQUIRED = EVIDENCE_FIELDS + ["complete_propositions", "regions_referenced",
                              "repeated_content_lemmas", "self_corrections"]

def groundedness(obj, text):
    """Share of quoted spans that literally occur in the transcript (NaN if no spans)."""
    low, hit, tot = text.lower(), 0, 0
    for f in EVIDENCE_FIELDS:
        for s in obj.get(f, []) or []:
            if isinstance(s, str) and s.strip():
                tot += 1
                hit += s.strip().lower() in low
    return hit / tot if tot else np.nan

def missing_fields(obj):
    return [f for f in REQUIRED if f not in obj]

def extract_one(text):
    return provider.text_features([text], prompt=V2_PROMPT)[0]

if RUN_LLM:
    calib = docs[docs.label.isna()].head(N_CALIB)
    probe = []
    for _, r in calib.iterrows():
        s = time.time()
        try:
            o = extract_one(r.text)
            probe.append({"file": r.file, "sec": time.time() - s, "ok": True,
                          "missing": missing_fields(o), "grounded": groundedness(o, r.text)})
        except Exception as e:
            probe.append({"file": r.file, "sec": time.time() - s, "ok": False,
                          "missing": None, "grounded": np.nan,
                          "err": f"{type(e).__name__}: {e}"[:120]})
    probe = pd.DataFrame(probe)
    print(probe.to_string(index=False))
    med = probe.sec.median()
    print(f"\nmedian {med:.1f}s/doc  ->  327 docs ~ {med*327/60:.0f} min")
    print(f"parsed OK: {probe.ok.sum()}/{len(probe)}   mean groundedness: {probe.grounded.mean():.2f}")
    print("\n>>> GATE: groundedness should stay > 0.80, same as run A.")
else:
    print("RUN_LLM = False -- skipping calibration.")


RUN_LLM = False -- skipping calibration.


## Block 5 — full MODEL_B extraction (resumable)

Same resume logic as notebook 08: one JSON line per doc, flushed immediately, already-done
files skipped on re-run.

In [6]:
# resumable JSONL extraction -- inlined. Each response is appended and flushed
# immediately; seal_last_line() repairs a crash-truncated final line before resume.
def load_raw(path):
    out = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                try:
                    rec = json.loads(line); out[rec["file"]] = rec
                except Exception:
                    pass
    return out

def seal_last_line(path):
    if path.exists() and path.stat().st_size:
        with path.open("rb+") as fh:
            fh.seek(-1, 2)
            if fh.read(1) != b"\n":
                fh.write(b"\n")

if RUN_LLM:
    seal_last_line(RAW_B)
    done = load_raw(RAW_B)
    todo = docs[~docs.file.isin(done)]
    print(f"already done: {len(done)}   remaining: {len(todo)}")
    t0 = time.time()
    with RAW_B.open("a", encoding="utf-8") as fh:
        for i, (_, r) in enumerate(todo.iterrows(), 1):
            rec = {"file": r.file, "label": (None if pd.isna(r.label) else int(r.label)),
                   "model": MODEL_B, "prompt_sha": PROMPT_SHA}
            try:
                rec["response"] = extract_one(r.text); rec["error"] = None
            except Exception as e:
                rec["response"] = None; rec["error"] = f"{type(e).__name__}: {e}"[:300]
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n"); fh.flush()
            if i % 10 == 0 or i == len(todo):
                el = time.time() - t0
                print(f"  {i}/{len(todo)}  {el/60:.1f} min, ~{el/i*(len(todo)-i)/60:.1f} min left", flush=True)

rawB = load_raw(RAW_B)
nB_ok = sum(v["response"] is not None for v in rawB.values())
print(f"\nMODEL_B extracted: {nB_ok}/{len(rawB)} ok, {len(rawB) - nB_ok} failed")

META_B.write_text(json.dumps({
    "model": MODEL_B, "base_url": BASE_URL, "temperature": TEMPERATURE, "max_tokens": MAX_TOKENS,
    "streaming": True, "prompt_version": "v2", "prompt_sha256_16": PROMPT_SHA,
    "prompt_chars": len(V2_PROMPT), "n_documents_sent": len(rawB), "n_succeeded": nB_ok,
    "n_failed": len(rawB) - nB_ok, "n_labelled_for_modelling": 241, "n_unlabelled_extracted": 86,
    "test_set_used": False,
    "run_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
}, indent=2))
print("metadata ->", META_B)



MODEL_B extracted: 327/327 ok, 0 failed
metadata -> fileDataset/outputs/v2b_run_metadata.json


## Block 6 — build both feature matrices

Identical `to_row` mapping as notebook 08. `dfA` = run A (`qwen3.5:122b`), `dfB` = run B
(`qwen3.6-35b`). Counts → per-100-word rates + two scale-free ratios.

In [7]:
# raw LLM JSON -> feature matrix -- inlined. Every count becomes a per-100-word
# rate; two scale-free ratios are added (specific/generic verb balance, region breadth).
LIST_FIELDS = {"n_entities": "named_entities", "n_specific_verb": "specific_action_verbs",
               "n_generic_verb": "generic_verbs", "n_locative": "locative_expressions",
               "n_hedge": "hedge_spans", "n_deictic": "deictic_spans",
               "n_metacomment": "metacomment_spans", "n_diminutive": "diminutive_or_affective_forms",
               "n_quantity": "quantity_expressions"}
INT_FIELDS = {"n_proposition": "complete_propositions", "n_selfcorrect": "self_corrections"}

def to_row(obj):
    r = {}
    for out, key in LIST_FIELDS.items():
        v = obj.get(key) or []
        r[out] = len(v) if isinstance(v, list) else 0
    for out, key in INT_FIELDS.items():
        v = obj.get(key, 0)
        r[out] = int(v) if isinstance(v, (int, float)) else 0
    reg = obj.get("regions_referenced") or []
    r["n_region"] = len(set(reg)) if isinstance(reg, list) else 0
    rep = obj.get("repeated_content_lemmas") or []
    r["n_repeated_lemma"] = sum(1 for x in rep if isinstance(x, dict) and x.get("lemma"))
    r["repeat_mass"] = sum(int(x.get("count", 0)) for x in rep
                           if isinstance(x, dict) and str(x.get("count", "")).isdigit())
    return r

def build_df(path):
    raw = load_raw(path)
    ok = {f: v["response"] for f, v in raw.items() if v["response"] is not None}
    feat = pd.DataFrame([{"file": f, **to_row(o)} for f, o in ok.items()])
    d = docs.merge(feat, on="file", how="inner")
    counts = [c for c in feat.columns if c != "file"]
    for c in counts:
        d[c + "_r100"] = 100 * d[c] / d["n_word"].clip(lower=1)
    d["specific_verb_ratio"] = d.n_specific_verb / (d.n_specific_verb + d.n_generic_verb).clip(lower=1)
    d["region_breadth"] = d.n_region / 3.0
    return d, counts, [c + "_r100" for c in counts], ["specific_verb_ratio", "region_breadth"]

dfA, COUNTS, RATES, RATIOS = build_df(RAW_A)
dfB, _, _, _ = build_df(RAW_B)
print(f"run A (MODEL_A): {len(dfA)} docs   run B (MODEL_B): {len(dfB)} docs")
print(f"features: {len(COUNTS)} counts + {len(RATES)} rates + {len(RATIOS)} ratios")

# labelled views, aligned to the SAME file order
labA = dfA[dfA.label.notna()].copy(); labA["label"] = labA.label.astype(int)
labB = dfB[dfB.label.notna()].copy(); labB["label"] = labB.label.astype(int)
common_lab = sorted(set(labA.file) & set(labB.file))
labA = labA.set_index("file").loc[common_lab].reset_index()
labB = labB.set_index("file").loc[common_lab].reset_index()
y = labA.label.values
print(f"labelled docs present in BOTH runs: {len(common_lab)}  (pos {y.sum()}, neg {(y==0).sum()})")


run A (MODEL_A): 327 docs   run B (MODEL_B): 327 docs
features: 14 counts + 14 rates + 2 ratios
labelled docs present in BOTH runs: 241  (pos 70, neg 171)


## Block 7 — feature stability (run A vs run B)

Per feature, on every document present in both runs:

* **Spearman ρ** — rank agreement (the headline number).
* **ICC(2,1)** — two-way random effects, absolute agreement, single rating: penalises
  systematic offsets between models, not just rank disagreement.
* **exact-match** — share of docs where the two integer counts are identical.
* **median A / median B** — to see systematic level shifts.

In [8]:
from scipy import stats

# cross-model feature agreement -- inlined. Spearman rho is the headline; ICC(2,1)
# (two-way random effects, absolute agreement) catches a systematic level shift that
# Spearman alone would hide (this is exactly what happened with n_entities).
def icc_2_1(a, b):
    Y = np.column_stack([a, b]).astype(float)
    n, k = Y.shape
    gm = Y.mean()
    ssr = k * ((Y.mean(1) - gm) ** 2).sum()          # between documents
    ssc = n * ((Y.mean(0) - gm) ** 2).sum()          # between models
    sse = ((Y - gm) ** 2).sum() - ssr - ssc
    msr, msc = ssr / (n - 1), ssc / (k - 1)
    mse = sse / ((n - 1) * (k - 1)) if (n - 1) * (k - 1) else np.nan
    if not np.isfinite(mse):
        return np.nan
    denom = msr + (k - 1) * mse + k * (msc - mse) / n
    return np.nan if denom == 0 else (msr - mse) / denom

A = dfA.set_index("file"); B = dfB.set_index("file")
common = sorted(set(A.index) & set(B.index))
A, B = A.loc[common], B.loc[common]
print(f"documents compared: {len(common)}")

rows = []
for f in COUNTS + RATES + RATIOS:
    a = A[f].astype(float).values
    b = B[f].astype(float).values
    both = a.std() > 0 and b.std() > 0
    rho  = stats.spearmanr(a, b).correlation if both else np.nan
    pear = float(np.corrcoef(a, b)[0, 1]) if both else np.nan
    rows.append({"feature": f, "spearman": rho, "pearson": pear,
                 "icc21": icc_2_1(a, b) if (a.std() > 0 or b.std() > 0) else np.nan,
                 "exact_match": float(np.mean(np.isclose(a, b))) if f in COUNTS else np.nan,
                 "medA": float(np.median(a)), "medB": float(np.median(b)),
                 "mean_abs_diff": float(np.mean(np.abs(a - b)))})
stab = pd.DataFrame(rows).set_index("feature")
stab["stable"] = stab.spearman >= STABILITY_MIN_RHO   # NaN spearman -> False

pd.set_option("display.width", 200, "display.max_rows", 80)
print(stab.round(3).sort_values("spearman", ascending=False).to_string())

n_stable = int(stab.loc[COUNTS + RATIOS, "stable"].sum())
print(f"\ncount+ratio features stable (rho >= {STABILITY_MIN_RHO}): "
      f"{n_stable}/{len(COUNTS) + len(RATIOS)}")
print("median Spearman over count features:", round(stab.loc[COUNTS, 'spearman'].median(), 3))
print("UNSTABLE count/ratio features:", list(stab.loc[COUNTS + RATIOS].query("not stable").index))
stab.to_csv(OUT / "v2_stability.csv")


documents compared: 327
                       spearman  pearson  icc21  exact_match    medA    medB  mean_abs_diff  stable
feature                                                                                            
n_proposition             0.900    0.878  0.866        0.459  13.000  12.000          1.300    True
n_locative                0.881    0.888  0.882        0.367   7.000   7.000          1.205    True
n_quantity                0.878    0.839  0.839        0.777   0.000   0.000          0.370    True
n_quantity_r100           0.870    0.829  0.828          NaN   0.000   0.000          0.491    True
n_hedge                   0.811    0.814  0.808        0.624   1.000   1.000          0.584    True
n_proposition_r100        0.802    0.817  0.804          NaN  18.421  17.391          1.892    True
n_locative_r100           0.797    0.823  0.816          NaN   9.524  10.145          1.618    True
n_metacomment             0.791    0.673  0.669        0.758   1.000   0.000

## Block 8 — automatic degeneracy gate  (`zero_share > 0.70`)

Applied to the **labelled** docs, in **either** run. Dropped count features also lose their
`_r100` rate. This is the only automatic drop; the stability flag from Block 7 is carried as
metadata and used for a sensitivity arm, not for dropping.

In [9]:
# degeneracy gate (mechanical, applied from disk values): drop a count if it is
# zero on > ZERO_SHARE_MAX of the labelled docs in EITHER extraction run.
zsA = (labA[COUNTS] == 0).mean()
zsB = (labB[COUNTS] == 0).mean()
gate = pd.DataFrame({"zero_share_A": zsA, "zero_share_B": zsB})
gate["degenerate"] = (gate.zero_share_A > ZERO_SHARE_MAX) | (gate.zero_share_B > ZERO_SHARE_MAX)
gate["stable"] = stab.loc[COUNTS, "stable"]
print(gate.round(3).sort_values("zero_share_A", ascending=False).to_string())

DROP_COUNTS = gate.index[gate.degenerate].tolist()
KEPT_COUNTS = [c for c in COUNTS if c not in DROP_COUNTS]
KEPT_RATES  = [c + "_r100" for c in KEPT_COUNTS]
KEPT_RATIOS = list(RATIOS)                      # ratios are scale-free, keep both
print(f"\ndropped by degeneracy gate : {DROP_COUNTS}")
print(f"kept counts ({len(KEPT_COUNTS)})     : {KEPT_COUNTS}")

# V2 feature representations
V2_RATES  = KEPT_RATES + KEPT_RATIOS                                        # length-normalised, non-degenerate
V2_ALL    = KEPT_COUNTS + KEPT_RATES + KEPT_RATIOS                          # counts + rates (ablation)
V2_STABLE = ([c + "_r100" for c in KEPT_COUNTS if stab.loc[c, "stable"]]
             + [r for r in KEPT_RATIOS if stab.loc[r, "stable"]])          # stability-gated rates

# the frozen primary: stability-gated rates + surface stats
FROZEN_PRIMARY_FEATURES = V2_STABLE + SURFACE

# audit: rebuild V2_STABLE from v2_stability.csv (the gate's own on-disk output) and
# confirm the in-memory list matches, and that it drops exactly the two unstable feats.
_sd = pd.read_csv(OUT / "v2_stability.csv", index_col="feature")
_v2s_disk = ([c + "_r100" for c in KEPT_COUNTS if bool(_sd.loc[c, "stable"])]
             + [r for r in KEPT_RATIOS if bool(_sd.loc[r, "stable"])])
assert _v2s_disk == V2_STABLE, (_v2s_disk, V2_STABLE)
assert "specific_verb_ratio" not in FROZEN_PRIMARY_FEATURES
assert "n_generic_verb_r100" not in FROZEN_PRIMARY_FEATURES
assert len(FROZEN_PRIMARY_FEATURES) == len(V2_STABLE) + len(SURFACE) == 20
assert [f for f in (V2_RATES + SURFACE) if f not in FROZEN_PRIMARY_FEATURES] == \
    ["n_generic_verb_r100", "specific_verb_ratio"]

print(f"\nV2_RATES  ({len(V2_RATES)}): {V2_RATES}")
print(f"V2_STABLE ({len(V2_STABLE)}): {V2_STABLE}")
print(f"FROZEN_PRIMARY_FEATURES ({len(FROZEN_PRIMARY_FEATURES)}): V2_STABLE + SURFACE")


                  zero_share_A  zero_share_B  degenerate  stable
n_entities               0.946         0.386        True   False
n_selfcorrect            0.622         0.556       False    True
n_quantity               0.544         0.519       False    True
n_metacomment            0.510         0.535       False    True
n_hedge                  0.344         0.423       False    True
n_deictic                0.261         0.423       False    True
n_generic_verb           0.183         0.154       False   False
n_region                 0.087         0.108       False    True
n_diminutive             0.083         0.108       False    True
n_proposition            0.029         0.029       False    True
n_locative               0.025         0.021       False    True
n_specific_verb          0.017         0.012       False    True
n_repeated_lemma         0.012         0.004       False    True
repeat_mass              0.012         0.004       False    True

dropped by degeneracy ga

## Block 9 — classifier comparison

Same CV protocol as notebooks 04–08: **10× repeated stratified 5-fold**, out-of-fold
probabilities averaged over repeats, AUC as the primary metric with a bootstrap 95% CI, plus
**paired** bootstrap Δ between the representations that matter for the thesis question.

* **Primary classifier — Elastic-Net logistic regression** (`LogisticRegressionCV`,
  `penalty="elasticnet"`, `solver="saga"`, `l1_ratio` and `C` tuned by inner CV inside every
  outer fold — no leakage).
* **Robustness — plain L2 logistic regression and Linear SVM** (SVM calibrated to
  probabilities via inner CV) on the representations of interest.
* All vectorisers / encoders sit **inside** the pipeline, so nothing is fitted on the whole
  set before CV.

Features come from **run A** (the reference run the frozen pipeline will consume).

In [10]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from helpers import (cv_proba as _cv_proba, boot_ci as _boot_ci,
                     evaluate as _evaluate, paired_bootstrap,
                     N_REPEATS_DEFAULT as N_REPEATS, N_SPLITS_DEFAULT as N_SPLITS)

# classifier factories -- 09-specific model choices, kept local
def enet():
    return make_pipeline(StandardScaler(), LogisticRegressionCV(
        penalty="elasticnet", solver="saga", l1_ratios=[0.3, 0.6, 0.9], Cs=8,
        max_iter=5000, class_weight="balanced", scoring="roc_auc",
        cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=-1))

def l2():
    return make_pipeline(StandardScaler(),
                         LogisticRegression(penalty="l2", max_iter=2000, class_weight="balanced"))

def lsvm():
    return make_pipeline(StandardScaler(), CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", max_iter=5000), method="sigmoid", cv=3))

def cv_proba(model, X):
    return _cv_proba(model, X, y, seed=SEED)

def boot_ci(p, n=2000):
    return _boot_ci(y, p, n=n, seed=SEED)

PROBA, res = {}, []
def run(name, clf_name, model, X, key):
    r = _evaluate(name, X, y, seed=SEED, model=model)
    PROBA[key] = r["_proba"]
    res.append({"key": key, "representation": name, "clf": clf_name, "n_feat": r["n_feat"],
                "AUC": r["AUC"], "CI_low": r["CI_low"], "CI_high": r["CI_high"],
                "macroF1": r["macroF1"], "balAcc": r["balAcc"]})
    print(f"  {name:38s} [{clf_name:4s}]  AUC {r['AUC']:.3f}  CI {r['CI_low']:.3f}-{r['CI_high']:.3f}")

X_len, X_surf = labA[["n_word"]].values, labA[SURFACE].values
FP = "V2 stable + surface (frozen primary)"

_ohe_v1 = None
v1p = next((p for p in [Path("../OutputsQwen/train_all_feature_values.csv"),
                        Path("OutputsQwen/train_all_feature_values.csv")] if p.exists()), None)
if v1p is not None:
    v1 = pd.read_csv(v1p).rename(columns={"File": "file"})
    v1c = [c for c in v1.columns if c not in ("file", "Class", "raw_llm_output")]
    m1 = labA[["file"]].merge(v1[["file"] + v1c], on="file", how="left").fillna("missing")
    _ohe_v1 = make_pipeline(OneHotEncoder(handle_unknown="ignore"),
                            LogisticRegression(max_iter=2000, class_weight="balanced"))

_tfidf = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True),
    LogisticRegression(max_iter=3000, class_weight="balanced"))

# ---- main comparison arms (ElasticNet) + the TF-IDF reference ceiling ----
print("ElasticNet:")
run("Transcript length", "enet", enet(), X_len,  "len")
run("Surface stats",     "enet", enet(), X_surf, "surf")
if _ohe_v1 is not None:
    run("V1 categorical", "enet", _ohe_v1, m1[v1c], "v1")
run(FP, "enet", enet(), labA[FROZEN_PRIMARY_FEATURES].values, "v2fp")
run("TF-IDF char 3-5gram (ceiling)", "l2", _tfidf, labA.text.values, "tfidf")

# ---- encoding-ablation arms: computed for the paired contrasts + the Block 9b
#      reproducibility check, NOT shown as table rows ----
run("V2 rates",           "enet", enet(), labA[V2_RATES].values,           "v2r")
run("V2 counts+rates",    "enet", enet(), labA[V2_ALL].values,             "v2all")
run("V2 stable-only",     "enet", enet(), labA[V2_STABLE].values,          "v2stab")
run("V2 rates + surface", "enet", enet(), labA[V2_RATES + SURFACE].values, "v2r_surf")

# ---- robustness: surface + frozen primary under L2 and linear SVM ----
print("\nRobustness (L2, linear SVM):")
for cn, ctor in [("l2", l2), ("svm", lsvm)]:
    run("Surface stats", cn, ctor(), X_surf, f"surf_{cn}")
    run(FP,             cn, ctor(), labA[FROZEN_PRIMARY_FEATURES].values, f"v2fp_{cn}")

tbl = pd.DataFrame(res)

# §12: one main table (5 arms, ElasticNet; TF-IDF = reference ceiling) ...
_main_keys = ["len", "surf", "v1", "v2fp", "tfidf"]
main_tbl = (tbl[tbl.key.isin(_main_keys)]
            .set_index("key").reindex([k for k in _main_keys if k in set(tbl.key)]).reset_index())
print("\n=== Main results — ElasticNet (TF-IDF is a reference ceiling, not a competitor) ===")
print(main_tbl[["representation", "n_feat", "AUC", "CI_low", "CI_high", "macroF1", "balAcc"]]
      .round(3).to_string(index=False))

# ... and one robustness table (surface + frozen primary x ElasticNet/L2/SVM)
_rob_keys = ["surf", "v2fp", "surf_l2", "v2fp_l2", "surf_svm", "v2fp_svm"]
rob = (tbl[tbl.key.isin(_rob_keys)]
       .set_index("key").reindex(_rob_keys).reset_index())
print("\n=== Robustness — same two arms under ElasticNet / L2 / linear SVM "
      "(classifier-sensitivity check, not extra competitors) ===")
print(rob[["representation", "clf", "n_feat", "AUC", "CI_low", "CI_high"]]
      .round(3).to_string(index=False))
print("\nEncoding ablation (not tabled): V2 rates alone / V2 counts+rates / V2_STABLE alone "
      "/ V2 rates+surface -- rates beat raw counts because counts still embed length; "
      "see the paired contrasts below and v2_stability_comparison.csv.")

tbl.sort_values("AUC", ascending=False).reset_index(drop=True).to_csv(
    OUT / "v2_stability_comparison.csv", index=False)


ElasticNet:


  Transcript length                      [enet]  AUC 0.691  CI 0.615-0.762


  Surface stats                          [enet]  AUC 0.743  CI 0.673-0.803


  V1 categorical                         [enet]  AUC 0.693  CI 0.617-0.766


  V2 stable + surface (frozen primary)   [enet]  AUC 0.806  CI 0.739-0.867


...

  V2 rates + surface                     [enet]  AUC 0.805  CI 0.737-0.868

Robustness (L2, linear SVM):


  Surface stats                          [l2  ]  AUC 0.742  CI 0.673-0.803


  V2 stable + surface (frozen primary)   [l2  ]  AUC 0.789  CI 0.719-0.853


  Surface stats                          [svm ]  AUC 0.749  CI 0.681-0.809


  V2 stable + surface (frozen primary)   [svm ]  AUC 0.787  CI 0.717-0.851

=== Main results — ElasticNet (TF-IDF is a reference ceiling, not a competitor) ===
                      representation n_feat   AUC  CI_low  CI_high  macroF1  balAcc
                   Transcript length      1 0.691   0.615    0.762    0.617   0.647
                       Surface stats      7 0.743   0.673    0.803    0.642   0.662
                      V1 categorical     10 0.693   0.617    0.766    0.614   0.625
V2 stable + surface (frozen primary)     20 0.806   0.739    0.867    0.744   0.756
       TF-IDF char 3-5gram (ceiling)   text 0.867   0.816    0.912    0.758   0.753

=== Robustness — same two arms under ElasticNet / L2 / linear SVM (classifier-sensitivity check, not extra competitors) ===
                      representation  clf n_feat   AUC  CI_low  CI_high
                       Surface stats enet      7 0.743   0.673    0.803
V2 stable + surface (frozen primary) enet     20 0.806   0.739    0

### Paired bootstrap — the decisive contrasts

In [11]:
# paired bootstrap on the SAME per-document OOF probabilities (helpers.paired_bootstrap,
# keyed through PROBA). Use this for every model comparison -- never eyeball overlapping CIs.
def paired(a, b, n=5000):
    return paired_bootstrap(PROBA[a], PROBA[b], y, n=n, seed=SEED)

CONTRASTS = [
    ("v2fp",  "surf", "V2 stable+surface (frozen primary) vs surface   (does V2 add to surface?)"),
    ("v2r",   "surf", "V2 rates            vs  surface alone"),
    ("v2r",   "len",  "V2 rates            vs  transcript length"),
    ("v2r",   "v1",   "V2 rates            vs  V1 categorical"),
    ("tfidf", "v2r",  "TF-IDF              vs  V2 rates"),
    ("v2stab","surf", "V2 stable-only      vs  surface alone"),
    ("v2fp_l2",  "surf_l2",  "[L2]  frozen primary vs surface"),
    ("v2fp_svm", "surf_svm", "[SVM] frozen primary vs surface"),
]
print(f"{'contrast':62s} {'dAUC':>8s}  {'95% CI':>18s}  {'P(d<=0)':>8s}")
for a, b, label in CONTRASTS:
    if a in PROBA and b in PROBA:
        m, ci, pneg = paired(a, b)
        flag = "  *" if (ci[0] > 0 or ci[1] < 0) else ""
        print(f"{label:62s} {m:+.4f}  [{ci[0]:+.4f},{ci[1]:+.4f}]  {pneg:6.3f}{flag}")


contrast                                                           dAUC              95% CI   P(d<=0)


V2 stable+surface (frozen primary) vs surface   (does V2 add to surface?) +0.0633  [+0.0046,+0.1205]   0.019  *


V2 rates            vs  surface alone                          +0.0251  [-0.0417,+0.0894]   0.225


V2 rates            vs  transcript length                      +0.0770  [-0.0094,+0.1638]   0.040


V2 rates            vs  V1 categorical                         +0.0749  [-0.0062,+0.1563]   0.037


TF-IDF              vs  V2 rates                               +0.0998  [+0.0365,+0.1655]   0.001  *


V2 stable-only      vs  surface alone                          +0.0326  [-0.0345,+0.0968]   0.160


[L2]  frozen primary vs surface                                +0.0474  [-0.0162,+0.1101]   0.075


[SVM] frozen primary vs surface                                +0.0382  [-0.0230,+0.0976]   0.107


## Block 9b — corrected frozen-primary feature set (stability-gated)

Block 10 froze `frozen_primary = V2_RATES + SURFACE`, which still carries two features Block 7/8 flagged as model-dependent: `specific_verb_ratio` (Spearman ρ = 0.44, BH-significant only under 122b) and `n_generic_verb_r100` (ρ = 0.34; gated out via its count `n_generic_verb`, ρ = 0.42). The frozen primary is corrected here to `V2_STABLE + SURFACE` **before** any test-set code. Same Block 9 protocol — 10×5 repeated CV, identical folds/seed, leak-free pipelines. The 61-case test set is not read.

In [12]:
# FROZEN_PRIMARY_FEATURES is defined in Block 8; its enet CV arm ("v2fp") ran in
# Block 9. This block: (a) the reproducibility guard, (b) the decisive paired bootstrap.
row_fp = next(r for r in res if r["representation"] == "V2 stable + surface (frozen primary)")
print(f"frozen primary: {len(FROZEN_PRIMARY_FEATURES)} cols  CV AUC {row_fp['AUC']:.4f}  "
      f"CI {row_fp['CI_low']:.3f}-{row_fp['CI_high']:.3f}")

# reproducibility guard: 'V2 stable-only' (13 feats, no surface) is the published
# 0.776 arm. Re-derive its AUC on the same folds and require it -- and the frozen
# primary (a superset) -- to land inside that arm's reported bootstrap CI band.
cmp_disk = pd.read_csv(OUT / "v2_stability_comparison.csv")
_ref = cmp_disk[(cmp_disk.representation == "V2 stable-only") & (cmp_disk.clf == "enet")].iloc[0]
_auc_stab_only = roc_auc_score(y, PROBA["v2stab"])
print(f"\n  reproduce 'V2 stable-only' : fresh AUC {_auc_stab_only:.4f}  vs reported "
      f"{_ref.AUC:.4f}   band [{_ref.CI_low:.3f}, {_ref.CI_high:.3f}]")
assert _ref.CI_low <= _auc_stab_only <= _ref.CI_high, (
    f"stable-only CV AUC {_auc_stab_only:.4f} outside reported CI band -- STOP, do not freeze")
assert _ref.CI_low <= row_fp["AUC"] <= _ref.CI_high, (
    f"frozen-primary CV AUC {row_fp['AUC']:.4f} outside stable-only CI band -- STOP, do not freeze")
print(f"  frozen primary in band     : AUC {row_fp['AUC']:.4f}  -> OK")

# --- the decisive paired bootstrap: V2_STABLE + surface  vs  surface alone ---
_m, _ci, _pneg = paired("v2fp", "surf", n=2000)
PAIRED_FROZEN_VS_SURFACE = {
    "contrast": "V2_STABLE + surface (enet)  vs  surface alone (enet)",
    "n_resamples": 2000, "delta_auc": float(_m),
    "ci95": [float(_ci[0]), float(_ci[1])], "p_delta_le_0": float(_pneg),
}
_mo, _cio, _pnego = paired("v2r_surf", "surf", n=2000)   # V2_RATES+surface, for context
PAIRED_RATES_SURF_VS_SURFACE = {
    "contrast": "V2_RATES + surface (enet)  vs  surface alone (enet)  [un-gated feature set]",
    "n_resamples": 2000, "delta_auc": float(_mo),
    "ci95": [float(_cio[0]), float(_cio[1])], "p_delta_le_0": float(_pnego),
}
print("\npaired bootstrap (2000 resamples, same folds as Block 9):")
print(f"  V2_STABLE+surface vs surface : dAUC {_m:+.4f}  95% CI [{_ci[0]:+.4f}, {_ci[1]:+.4f}]"
      f"  P(dAUC<=0) = {_pneg:.3f}{'   *' if _ci[0] > 0 else ''}")
print(f"  (context) V2_RATES+surface   : dAUC {_mo:+.4f}  95% CI [{_cio[0]:+.4f}, {_cio[1]:+.4f}]"
      f"  P(dAUC<=0) = {_pnego:.3f}")
print("\n>>> decisive number for 'does V2 add beyond surface stats':")
print(f"    dAUC = {_m:+.4f}   95% CI [{_ci[0]:+.4f}, {_ci[1]:+.4f}]   P(dAUC<=0) = {_pneg:.3f}")


frozen primary: 20 cols  CV AUC 0.8058  CI 0.739-0.867

  reproduce 'V2 stable-only' : fresh AUC 0.7754  vs reported 0.7754   band [0.699, 0.847]
  frozen primary in band     : AUC 0.8058  -> OK



paired bootstrap (2000 resamples, same folds as Block 9):
  V2_STABLE+surface vs surface : dAUC +0.0635  95% CI [+0.0065, +0.1197]  P(dAUC<=0) = 0.016   *
  (context) V2_RATES+surface   : dAUC +0.0628  95% CI [+0.0059, +0.1183]  P(dAUC<=0) = 0.017

>>> decisive number for 'does V2 add beyond surface stats':
    dAUC = +0.0635   95% CI [+0.0065, +0.1197]   P(dAUC<=0) = 0.016


## Block 10 — freeze the pipeline

Everything needed to reproduce the result and to run the **single** future test-set
evaluation is written to `fileDataset/outputs/frozen_pipeline_v2/`:

* `frozen_spec.json` — models, prompt SHA, gate thresholds, feature lists, CV protocol,
  classifier params, stability + comparison summaries.
* `pipe_<name>.joblib` — the fitted pipelines (surface baseline, V2 rates + surface, TF-IDF
  ceiling), trained on all labelled run-A docs.

The 61 test transcripts are **not** read here.

In [13]:
import joblib, sklearn

# freeze the fitted pipelines + a spec file. The artifact is a dict carrying the
# exact input-column list so it can be re-checked on reload (see Block 10b).
def freeze(pipe, path, feats):
    joblib.dump({"pipeline": pipe, "features": list(feats),
                 "feature_source_run": "A / " + MODEL_A}, path)

_tfidf_pipe = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True),
    LogisticRegression(max_iter=3000, class_weight="balanced"))

FROZEN = {
    "surface":           (l2(),   SURFACE,                 labA[SURFACE].values),
    "v2_stable_surface": (enet(), FROZEN_PRIMARY_FEATURES, labA[FROZEN_PRIMARY_FEATURES].values),
    "tfidf_char35":      (_tfidf_pipe, "text",             labA.text.values),
}
for name, (model, feats, X) in FROZEN.items():
    model.fit(X, y)
    freeze(model, FROZEN_DIR / f"pipe_{name}.joblib", feats)
    print("froze", name)

# deprecate (never delete) any superseded V2_RATES+SURFACE primary from an earlier run
_old = FROZEN_DIR / "pipe_v2_rates_surface.joblib"
_dep = FROZEN_DIR / "pipe_v2_rates_surface_deprecated.joblib"
_dep_reason = ("superseded by pipe_v2_stable_surface.joblib: carried features flagged unstable "
               "in Block 7/8 (specific_verb_ratio rho=0.44, n_generic_verb_r100 rho=0.34)")
if _old.exists():
    _old.replace(_dep); print("deprecated existing artifact ->", _dep.name)
elif not _dep.exists():
    _m = enet(); _m.fit(labA[V2_RATES + SURFACE].values, y)
    freeze(_m, _dep, V2_RATES + SURFACE); print("wrote audit copy ->", _dep.name)
else:
    print("deprecated audit copy already present ->", _dep.name)

MODEL_ARTIFACT = "pipe_v2_stable_surface.joblib"

spec = {
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "sklearn_version": sklearn.__version__,
    "prompt_sha256_16": PROMPT_SHA,
    "prompt_sha256_assert": EXPECTED_PROMPT_SHA,
    "models": {"A": MODEL_A, "B": MODEL_B},
    "test_extraction_model": MODEL_A,     # qwen3.5:122b -- the run the frozen pipe was trained on
    "model_artifact": MODEL_ARTIFACT,
    "deprecated_artifacts": {_dep.name: _dep_reason},
    "llm_params": {"temperature": TEMPERATURE, "max_tokens": MAX_TOKENS, "stream": True, "think": False},
    "gates": {"zero_share_max": ZERO_SHARE_MAX, "stability_min_rho": STABILITY_MIN_RHO},
    "dropped_degenerate": DROP_COUNTS,
    "unstable_features": list(stab.loc[COUNTS + RATIOS].query("not stable").index),
    "feature_sets": {"V2_RATES": V2_RATES, "V2_STABLE": V2_STABLE, "SURFACE": SURFACE,
                     "frozen_primary": FROZEN_PRIMARY_FEATURES},
    "cv_protocol": {"scheme": "repeated stratified k-fold", "repeats": N_REPEATS,
                    "folds": N_SPLITS, "seed": SEED, "metric": "roc_auc",
                    "ci": "2000x subject bootstrap"},
    "classifiers": {
        "primary": "LogisticRegressionCV(penalty=elasticnet, solver=saga, "
                   "l1_ratios=[.3,.6,.9], Cs=8, class_weight=balanced), StandardScaler",
        "robustness": ["LogisticRegression(l2, class_weight=balanced)",
                       "CalibratedClassifierCV(LinearSVC(class_weight=balanced))"]},
    "stability_summary": {
        "docs_compared": len(common),
        "median_spearman_counts": float(stab.loc[COUNTS, "spearman"].median()),
        "n_stable_count_ratio": int(stab.loc[COUNTS + RATIOS, "stable"].sum()),
        "n_total_count_ratio": len(COUNTS) + len(RATIOS)},
    "frozen_primary_cv_auc": float(row_fp["AUC"]),
    "paired_bootstrap": {
        "frozen_primary_vs_surface": PAIRED_FROZEN_VS_SURFACE,
        "rates_surface_vs_surface": PAIRED_RATES_SURF_VS_SURFACE},
    "comparison_auc": tbl.set_index(["representation", "clf"]).AUC.round(4).to_dict().__repr__(),
    "test_set_used": False,
}
# preserve a test-set seal: if notebook 10 has already run, carry its
# test_set_used / test_results forward -- re-running 09 must NOT un-seal.
_spec_path = FROZEN_DIR / "frozen_spec.json"
if _spec_path.exists():
    _prev = json.loads(_spec_path.read_text())
    if _prev.get("test_set_used") is True and "test_results" in _prev:
        spec["test_set_used"]     = True
        spec["test_set_used_utc"] = _prev.get("test_set_used_utc")
        spec["test_results"]      = _prev["test_results"]
        print("NOTE: existing test-set seal preserved (notebook 10 has already run)")
_spec_path.write_text(json.dumps(spec, indent=2, default=str))
print("\nfrozen_spec.json ->", FROZEN_DIR / "frozen_spec.json")
print("frozen_primary :", len(spec["feature_sets"]["frozen_primary"]), "cols  ->", MODEL_ARTIFACT)
_pb = PAIRED_FROZEN_VS_SURFACE
print(f"paired dAUC    : {_pb['delta_auc']:+.4f}  CI {_pb['ci95']}  P(<=0) {_pb['p_delta_le_0']:.3f}")


froze surface


froze v2_stable_surface
froze tfidf_char35
deprecated audit copy already present -> pipe_v2_rates_surface_deprecated.joblib
NOTE: existing test-set seal preserved (notebook 10 has already run)

frozen_spec.json -> fileDataset/outputs/frozen_pipeline_v2/frozen_spec.json
frozen_primary : 20 cols  -> pipe_v2_stable_surface.joblib
paired dAUC    : +0.0635  CI [0.006543441539207828, 0.11973858587914607]  P(<=0) 0.016


## Block 10b — final consistency check

Reload `frozen_spec.json` and `pipe_v2_stable_surface.joblib` from disk and assert they agree — same reload-and-assert pattern as Block 0–1 (`assert RAW_A.exists()`, `assert PROMPT_SHA == EXPECTED_PROMPT_SHA`). No test-set code.

In [14]:
# reload spec + artifact fresh and assert they agree -- the checks assert_consistent()
# used to bundle, now written out so a reader sees exactly what is guaranteed.
_spec = json.loads((FROZEN_DIR / "frozen_spec.json").read_text())
_art  = joblib.load(FROZEN_DIR / _spec["model_artifact"])
_fp   = _spec["feature_sets"]["frozen_primary"]
_pipe = _art["pipeline"]

assert _pipe.n_features_in_ == len(_fp), (
    f"pipeline expects {_pipe.n_features_in_} cols, frozen_primary has {len(_fp)}")
assert list(_art["features"]) == list(_fp), "artifact feature list != spec frozen_primary"
assert list(_fp) == list(V2_STABLE) + list(SURFACE), "frozen_primary != V2_STABLE + SURFACE"
assert "specific_verb_ratio" not in _fp and "n_generic_verb_r100" not in _fp
_sha = hashlib.sha256(V2_PROMPT.encode()).hexdigest()[:16]
assert _sha == EXPECTED_PROMPT_SHA == _spec["prompt_sha256_16"] == "f52aae5d939d620e"
assert _spec["test_extraction_model"] == MODEL_A == "qwen3.5:122b"
assert _spec["model_artifact"] == "pipe_v2_stable_surface.joblib"
assert _spec["dropped_degenerate"] == ["n_entities"]
assert _spec["unstable_features"] == ["n_entities", "n_generic_verb", "specific_verb_ratio"]
assert (FROZEN_DIR / "pipe_v2_rates_surface_deprecated.joblib").exists(), "audit copy missing"
assert not (FROZEN_DIR / "pipe_v2_rates_surface.joblib").exists(), "stale non-deprecated primary present"
# either freshly frozen (test not yet used) or a seal preserved from notebook 10
assert (_spec["test_set_used"] is False) or (
    _spec["test_set_used"] is True and "test_results" in _spec), _spec["test_set_used"]

print("consistency check -- frozen spec + primary artifact")
print(f"  model_artifact        : {_spec['model_artifact']}")
print(f"  frozen_primary        : {len(_fp)} cols  == pipeline.n_features_in_ ({_pipe.n_features_in_})")
print(f"  excludes unstable     : specific_verb_ratio, n_generic_verb_r100  -> not present")
print(f"  prompt sha256[:16]    : {_sha}   (== EXPECTED_PROMPT_SHA, == spec)")
print(f"  test_extraction_model : {_spec['test_extraction_model']}")
print(f"  frozen_primary_cv_auc : {_spec['frozen_primary_cv_auc']:.4f}")
_pb = _spec["paired_bootstrap"]["frozen_primary_vs_surface"]
print(f"  paired dAUC (vs surf) : {_pb['delta_auc']:+.4f}  CI [{_pb['ci95'][0]:+.4f}, {_pb['ci95'][1]:+.4f}]"
      f"  P(<=0) {_pb['p_delta_le_0']:.3f}")
print(f"  test_set_used         : {_spec['test_set_used']}"      + ("  (seal preserved from notebook 10)" if _spec['test_set_used'] else ""))
print("\nALL CHECKS PASSED -- pipeline frozen, test set untouched.")


consistency check -- frozen spec + primary artifact
  model_artifact        : pipe_v2_stable_surface.joblib
  frozen_primary        : 20 cols  == pipeline.n_features_in_ (20)
  excludes unstable     : specific_verb_ratio, n_generic_verb_r100  -> not present
  prompt sha256[:16]    : f52aae5d939d620e   (== EXPECTED_PROMPT_SHA, == spec)
  test_extraction_model : qwen3.5:122b
  frozen_primary_cv_auc : 0.8058
  paired dAUC (vs surf) : +0.0635  CI [+0.0065, +0.1197]  P(<=0) 0.016
  test_set_used         : True  (seal preserved from notebook 10)

ALL CHECKS PASSED -- pipeline frozen, test set untouched.


## Block 11 — verdict

In [15]:
med_rho = stab.loc[COUNTS, "spearman"].median()
n_stab  = int(stab.loc[COUNTS + RATIOS, "stable"].sum()); n_tot = len(COUNTS) + len(RATIOS)
m, ci, pneg = paired("v2fp", "surf")

def auc_of(key): return roc_auc_score(y, PROBA[key])

print("Q1  Are the V2 features stable across LLMs?")
print(f"    median Spearman rho (counts) = {med_rho:.2f}")
print(f"    {n_stab}/{n_tot} count+ratio features stable at rho >= {STABILITY_MIN_RHO}")
print(f"    dropped as degenerate (zero_share > {ZERO_SHARE_MAX}): {DROP_COUNTS}")
print(f"    unstable: {list(stab.loc[COUNTS + RATIOS].query('not stable').index)}")
print()
print("Q2  Do V2 features add information beyond surface statistics?")
print(f"    surface alone                    AUC = {auc_of('surf'):.3f}")
print(f"    V2 stable + surface (frozen)     AUC = {auc_of('v2fp'):.3f}")
print(f"    paired dAUC = {m:+.3f}   95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}]   P(dAUC<=0) = {pneg:.3f}")
verdict = ("ADDS signal beyond surface (CI above 0)" if ci[0] > 0 else
           "does NOT add demonstrable signal beyond surface (CI crosses 0)")
print(f"    -> {verdict}")
print()
print(f"    TF-IDF ceiling AUC = {auc_of('tfidf'):.3f}   (gap to frozen primary = "
      f"{auc_of('tfidf') - auc_of('v2fp'):+.3f})")
print("\nTEST SET: untouched. One evaluation, once, after this pipeline is frozen.")


Q1  Are the V2 features stable across LLMs?
    median Spearman rho (counts) = 0.68
    13/16 count+ratio features stable at rho >= 0.6
    dropped as degenerate (zero_share > 0.7): ['n_entities']
    unstable: ['n_entities', 'n_generic_verb', 'specific_verb_ratio']

Q2  Do V2 features add information beyond surface statistics?
    surface alone                    AUC = 0.743
    V2 stable + surface (frozen)     AUC = 0.806
    paired dAUC = +0.063   95% CI [+0.005, +0.120]   P(dAUC<=0) = 0.019
    -> ADDS signal beyond surface (CI above 0)

    TF-IDF ceiling AUC = 0.867   (gap to frozen primary = +0.061)

TEST SET: untouched. One evaluation, once, after this pipeline is frozen.
